In [1]:
import numpy as np
from PIL import Image
import math

def reassemble_image(npy_path, output_image_path, target_width, target_height, chunk_width, chunk_height):
    """
    Reassembles an image from chunks stored in a .npy file.

    Args:
        npy_path (str): Path to the input .npy file.
        output_image_path (str): Path to save the reassembled image.
        target_width (int): The width the image was resized to *before* chunking.
        target_height (int): The height the image was resized to *before* chunking.
        chunk_width (int): The width of each chunk.
        chunk_height (int): The height of each chunk.
    """
    # --- 1. Load the NPY data ---
    # Expected shape: (1, total_chunks, chunk_dim) where chunk_dim = chunk_h * chunk_w * 3
    all_chunks_data = np.load(npy_path)
    print(f"Loaded array shape: {all_chunks_data.shape}")

    # Remove the first dimension (batch size = 1)
    if all_chunks_data.shape[0] == 1:
        all_chunks_data = all_chunks_data[0] # Shape: (total_chunks, chunk_dim)

    num_channels = 3 # Assuming RGB
    chunk_dim = chunk_width * chunk_height * num_channels
    if all_chunks_data.shape[1] != chunk_dim:
        raise ValueError(f"Chunk dimension mismatch. Expected {chunk_dim}, got {all_chunks_data.shape[1]}")

    total_chunks = all_chunks_data.shape[0]

    # --- 2. Calculate Padded Dimensions (same logic as in Rust) ---
    padded_w = math.ceil(target_width / chunk_width) * chunk_width
    padded_h = math.ceil(target_height / chunk_height) * chunk_height
    print(f"Calculated padded dimensions: {padded_w}x{padded_h}")

    num_chunks_x = padded_w // chunk_width
    num_chunks_y = padded_h // chunk_height

    if num_chunks_x * num_chunks_y != total_chunks:
         raise ValueError(f"Total chunks mismatch. Calculated {num_chunks_x * num_chunks_y}, found {total_chunks} in file.")

    # --- 3. Reshape chunks back into image format ---
    # Reshape flat chunks into (total_chunks, chunk_height, chunk_width, channels)
    chunks_reshaped = all_chunks_data.reshape(
        total_chunks, chunk_height, chunk_width, num_channels
    )

    # --- 4. Assemble the Padded Image ---
    # Create an empty array for the padded image
    padded_image_np = np.zeros((padded_h, padded_w, num_channels), dtype=np.uint8)

    chunk_idx = 0
    for cy in range(num_chunks_y):
        for cx in range(num_chunks_x):
            start_y = cy * chunk_height
            end_y = start_y + chunk_height
            start_x = cx * chunk_width
            end_x = start_x + chunk_width

            padded_image_np[start_y:end_y, start_x:end_x, :] = chunks_reshaped[chunk_idx]
            chunk_idx += 1

    # --- 5. Crop to Target Dimensions ---
    # Remove the padding added during the chunking process
    final_image_np = padded_image_np[:target_height, :target_width, :]
    print(f"Final image shape after cropping padding: {final_image_np.shape}")

    # --- 6. Convert to PIL Image and Save ---
    final_image = Image.fromarray(final_image_np, 'RGB')
    final_image.save(output_image_path)
    print(f"Reassembled image saved to: {output_image_path}")

# --- Example Usage ---
if __name__ == "__main__":
    # --- These parameters MUST match those used in the Rust program ---
    TARGET_WIDTH = 100
    TARGET_HEIGHT = 80
    CHUNK_WIDTH = 20
    CHUNK_HEIGHT = 20
    NPY_FILE = "output_chunks/image_chunks.npy" # Path to the output from Rust
    OUTPUT_IMAGE = "reassembled_image.png"
    # ------------------------------------------------------------------

    reassemble_image(NPY_FILE, OUTPUT_IMAGE, TARGET_WIDTH, TARGET_HEIGHT, CHUNK_WIDTH, CHUNK_HEIGHT)

ModuleNotFoundError: No module named 'PIL'